## Importing Library

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Loading Data

In [ ]:
dataset = pd.read_csv('Loan_Application.csv')

In [ ]:
dataset.head()

,CustomerNumber,CreditScore,City,Gender,Age,Home_Living_Since,Avg_Acc_Bal,Assets,First Loan,Co-applicant,Mon_Sal,Loan
0,1,619,Bangalore,Female,41,2,0.00,1,1,1,50674.440,1
1,2,608,Hyderabad,Female,40,1,41903.93,1,0,1,56271.290,0
2,3,502,Bangalore,Female,41,8,79830.40,3,1,0,56965.785,1
3,4,699,Bangalore,Female,38,1,0.00,2,0,0,46913.315,0
4,5,850,Hyderabad,Female,42,2,62755.41,1,1,1,39542.050,0


## Data Pre-processing

### Dummy Variables



In [ ]:
dataset_dummy = pd.get_dummies(dataset, drop_first=True)

In [ ]:
dataset_dummy.head()

,CustomerNumber,CreditScore,Age,Home_Living_Since,Avg_Acc_Bal,Assets,First Loan,Co-applicant,Mon_Sal,Loan,City_Chennai,City_Hyderabad,Gender_Male
0,1,619,41,2,0.00,1,1,1,50674.440,1,False,False,False
1,2,608,40,1,41903.93,1,0,1,56271.290,0,False,True,False
2,3,502,41,8,79830.40,3,1,0,56965.785,1,False,False,False
3,4,699,38,1,0.00,2,0,0,46913.315,0,False,False,False
4,5,850,42,2,62755.41,1,1,1,39542.050,0,False,True,False


In [ ]:
dataset_dummy.columns

Index(['CustomerNumber', 'CreditScore', 'Age', 'Home_Living_Since',
       'Avg_Acc_Bal', 'Assets', 'First Loan', 'Co-applicant', 'Mon_Sal',
       'Loan', 'City_Chennai', 'City_Hyderabad', 'Gender_Male'],
      dtype='object')

In [ ]:
X = dataset_dummy.loc[:, ['CreditScore', 'Age', 'Home_Living_Since',
       'Avg_Acc_Bal', 'Assets', 'First Loan', 'Co-applicant', 'Mon_Sal',
       'City_Chennai', 'City_Hyderabad', 'Gender_Male']].values

In [ ]:
y = dataset_dummy.loc[:, ['Loan']].values

### Training-Testing Splitting

In [ ]:
# Splitting the dataset into the Training set and Test set
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

### Normalization - Feature Standardization

In [ ]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Model Building - ANN


In [ ]:
# Importing the Keras libraries and packages
import keras
from keras.models import Sequential # Functional
from keras.layers import Dense
from keras.layers import Dropout

### NN Architecture

In [ ]:
# Initialising the ANN
classifier = Sequential()

# Adding the input layer and the first hidden layer
classifier.add(Dense(units = 15, activation = 'relu', input_dim = 11))

In [ ]:
# Adding regularization
# classifier.add(Dropout(rate = 0.1))

Dropout parameter of 0.1 means 10% of the neurons would be disabled at each iteration.
If required, increase by 0.1 to improve the accuracy until 0.5.
So, that means p = 1, means no neurons and that is under-fitting.
In general, don't go over 0.5.

In [ ]:
# Adding the second hidden layer
classifier.add(Dense(units = 6, activation = 'relu'))

In [ ]:
# Adding regularization
# classifier.add(Dropout(rate = 0.1))

In [ ]:
# Adding the output layer
classifier.add(Dense(units = 1, activation = 'sigmoid'))

In [ ]:
classifier.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 15)                180       
                                                                 
 dense_1 (Dense)             (None, 6)                 96        
                                                                 
 dense_2 (Dense)             (None, 1)                 7         
                                                                 
Total params: 283 (1.11 KB)
Trainable params: 283 (1.11 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


### Model Compilation

In [ ]:
# Compiling the ANN
classifier.compile(optimizer = 'sgd', loss = 'binary_crossentropy', metrics = ['accuracy'])

Optimizers

Optimizers adjust the model’s parameters based on the gradients of the loss function with respect to these parameters, using a process called gradient descent (or variants of it). Gradients point in the direction where the model should update its parameters to reduce error, and optimizers determine the step size for each update.

Key Concepts in Optimization:

	1.	Gradient: The derivative of the loss function with respect to the model’s parameters, showing how to adjust parameters to reduce error.
	2.	Learning Rate: A hyperparameter that controls how big of a step the optimizer takes in the direction of the gradient. A high learning rate can lead to overshooting the minimum, while a low rate may cause slow convergence.
	3.	Batch Size: The number of samples used to compute the gradients in each optimization step. It can be small (Stochastic Gradient Descent) or large (Batch Gradient Descent).


Types of Optimizers

- Stochastic Gradient Descent (SGD):
  - How it works: SGD updates the model's parameters using a small subset (batch) of the data instead of the entire dataset. It computes the gradients and updates parameters for each batch, which helps to speed up training
  - Advantages: Faster updates, can escape local minima.
  - Disadvantages: Can be noisy, may not converge as quickly.

- Momentum:
  - How it works: Momentum adds a fraction of the previous update to the current one, helping the model move faster in directions with consistent gradients and dampen oscillations.
  - Advantages: Speeds up convergence, reduces oscillations.
  - Disadvantages: Requires tuning the momentum parameter.

- RMSprop:
  - How it works: RMSprop divides the learning rate by an exponentially decaying average of squared gradients. This stabilizes the learning process by scaling the learning rate for each parameter individually.
  - Advantages: Prevents exploding gradients, good for non-stationary objectives.
  - Disadvantages: Requires tuning of the decay parameter.

- Adam (Adaptive Moment Estimation):
  - How it works: Adam combines the benefits of Momentum and RMSprop. It maintains two moving averages: one for the gradient (like momentum) and one for the squared gradient (like RMSprop), then it adapts the learning rate for each parameter.
  - Advantages: Adaptive learning rates, works well in practice.
  - Disadvantages: May require more memory and tuning, can sometimes not generalize as well.



### Model Fitting

In [ ]:
# Fitting the ANN to the Training set

classifier.fit(X_train, y_train, batch_size = 10, epochs = 5, validation_data=(X_test, y_test))

Epoch 1/5
800/800 [==============================] - 5s 4ms/step - loss: 0.5193 - accuracy: 0.7795 - val_loss: 0.4806 - val_accuracy: 0.7980
Epoch 2/5
800/800 [==============================] - 3s 3ms/step - loss: 0.4617 - accuracy: 0.7975 - val_loss: 0.4537 - val_accuracy: 0.7965
Epoch 3/5
800/800 [==============================] - 3s 3ms/step - loss: 0.4418 - accuracy: 0.8034 - val_loss: 0.4363 - val_accuracy: 0.8095
Epoch 4/5
800/800 [==============================] - 2s 3ms/step - loss: 0.4287 - accuracy: 0.8089 - val_loss: 0.4237 - val_accuracy: 0.8145
Epoch 5/5
800/800 [==============================] - 2s 3ms/step - loss: 0.4179 - accuracy: 0.8155 - val_loss: 0.4126 - val_accuracy: 0.8215


## Model Evaluation

In [ ]:
# Predicting the Test set results
y_pred = classifier.predict(X_test)
y_pred = (y_pred > 0.5)

63/63 [==============================] - 0s 1ms/step


In [ ]:
# Making the Confusion Matrix
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[1556   39]
 [ 318   87]]


## Early Stopping and Model Check Point

In [ ]:
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [ ]:
early_stoppping = EarlyStopping(monitor='val_loss', verbose=1, patience=10)
best_model = ModelCheckpoint('best_model.h5', monitor='val_acc', verbose=1, save_best_only=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2,
                              patience=5, min_lr=0.001)

In [ ]:
classifier.fit(X_train, y_train, batch_size = 10, epochs = 5, validation_data=(X_test, y_test), callbacks=[early_stoppping, best_model, reduce_lr ])

Epoch 1/5
796/800 [============================>.] - ETA: 0s - loss: 0.4052 - accuracy: 0.8253

800/800 [==============================] - 5s 6ms/step - loss: 0.4060 - accuracy: 0.8250 - val_loss: 0.4005 - val_accuracy: 0.8305 - lr: 0.0100
Epoch 2/5
787/800 [============================>.] - ETA: 0s - loss: 0.3945 - accuracy: 0.8320

800/800 [==============================] - 3s 4ms/step - loss: 0.3938 - accuracy: 0.8329 - val_loss: 0.3866 - val_accuracy: 0.8380 - lr: 0.0100
Epoch 3/5
782/800 [============================>.] - ETA: 0s - loss: 0.3799 - accuracy: 0.8387

800/800 [==============================] - 3s 4ms/step - loss: 0.3806 - accuracy: 0.8386 - val_loss: 0.3743 - val_accuracy: 0.8450 - lr: 0.0100
Epoch 4/5
800/800 [==============================] - ETA: 0s - loss: 0.3704 - accuracy: 0.8465

800/800 [==============================] - 3s 4ms/step - loss: 0.3704 - accuracy: 0.8465 - val_loss: 0.3660 - val_accuracy: 0.8500 - lr: 0.0100
Epoch 5/5
785/800 [============================>.] - ETA: 0s - loss: 0.3605 - accuracy: 0.8518

800/800 [==============================] - 2s 3ms/step - loss: 0.3625 - accuracy: 0.8504 - val_loss: 0.3624 - val_accuracy: 0.8510 - lr: 0.0100


## Grid Search CV

In [ ]:
!pip install scikeras

In [ ]:
from scikeras.wrappers import KerasClassifier

from sklearn.model_selection import GridSearchCV
from keras.models import Sequential
from keras.layers import Dense

def build_classifier():
    classifier = Sequential()

    classifier.add(Dense(units = 15, activation = 'relu', input_dim = 11))
    classifier.add(Dense(units = 10, activation = 'relu'))
    #classifier.add(Dropout(p = 0.1))
    classifier.add(Dense(units = 1, activation = 'sigmoid'))
    #classifier.add(Dropout(p = 0.1))
    classifier.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])
    return classifier

classifier = KerasClassifier(model = build_classifier)

parameters = {'batch_size': [100, 300],
              #'epochs': [100, 500],
              'epochs': [2, 3],
              'optimizer': ['adam', 'rmsprop']}

grid_search = GridSearchCV(estimator = classifier,
                           param_grid = parameters,
                           scoring = 'accuracy',
                            cv=2) #10

In [ ]:
grid_search = grid_search.fit(X_train, y_train)

Epoch 1/2
40/40 [==============================] - 1s 4ms/step - loss: 0.7014 - accuracy: 0.5238
Epoch 2/2
40/40 [==============================] - 0s 2ms/step
Epoch 1/2
40/40 [==============================] - 1s 3ms/step - loss: 0.6563 - accuracy: 0.6285
Epoch 2/2
40/40 [==============================] - 0s 2ms/step
Epoch 1/2
40/40 [==============================] - 1s 3ms/step - loss: 0.7624 - accuracy: 0.4053
Epoch 2/2
40/40 [==============================] - 0s 2ms/step
Epoch 1/2
40/40 [==============================] - 1s 3ms/step - loss: 0.6872 - accuracy: 0.5803
Epoch 2/2
40/40 [==============================] - 0s 2ms/step
Epoch 1/3
40/40 [==============================] - 1s 3ms/step - loss: 0.6548 - accuracy: 0.6327
Epoch 2/3
40/40 [==============================] - 0s 3ms/step - loss: 0.5543 - accuracy: 0.7772
Epoch 3/3
40/40 [==============================] - 0s 2ms/step
Epoch 1/3
40/40 [==============================] - 1s 3ms/step - loss: 0.5307 - accuracy: 0.7960
Epoch 

In [ ]:
best_parameters = grid_search.best_params_
best_accuracy = grid_search.best_score_

print('best_parameters: ', best_parameters)
print('best accuracy: ', best_accuracy)

best_parameters:  {'batch_size': 100, 'epochs': 3, 'optimizer': 'adam'}
best accuracy:  0.796
